In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
block_size = 32
batch_size = 16
vocab_size = 81
max_iters = 3000
eval_iters = 200
learning_rate =3e-4
n_embd = 384
n_layers = 2
n_head = 2
dropout = 0.2

##### If your pc has higher specs you can increase the parameters to get better results 

In [2]:
print(device)

cuda


In [3]:
with open("../data/wizard_of_oz.txt",'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))

81

 !"&'()*,-.0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz﻿


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l]) 

In [6]:
data = torch.tensor(encode(text),dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(0,len(data)-block_size,size=(batch_size,)) # block_size=8 , batch_size=4
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [8]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd,head_size,bias=False)
        self.query = nn.Linear(n_embd,head_size,bias=False)
        self.value = nn.Linear(n_embd,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        # computing attention scores
        wei = torch.matmul(q,k.transpose(-2,-1)) * (k.shape[-1]**0.5) # (B,T,hs) @ (B,hs,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0,float('-inf'))
        wei = F.softmax(wei,dim=-1)
        # perform weighted aggregation
        v = self.value(x)
        out = wei @ v #(B,T,T) @ (B,T,hs) -> (B,T,hs)
        return out

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self,n_head,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_head*head_size,n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads],dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [10]:
class FeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self,x):
        return self.net(x)

In [11]:
class Block(nn.Module):
    def __init__(self,n_embd,n_head):
        super().__init__()
        head_size = n_embd//n_head
        self.sa = MultiHeadAttention(n_head,head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x+y) 
        return x
        

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layers)]) # Decoder blocks
        self.ln_f = nn.LayerNorm(n_embd) # final layer normalizer
        self.lm_head = nn.Linear(n_embd,vocab_size)
        self.apply(self.__init__weights)

    def __init__weights(self,module):
        if isinstance(module,nn.Linear):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)             

    def forward(self,index,targets=None):
        B, T = index.shape
        tok_embds = self.token_embedding_table(index) # (B,T,C)
        pos_embds = self.position_embedding_table(torch.arange(T,device=device)) # (T,C)
        x = pos_embds+tok_embds # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits,loss
    
    def generate(self,index,max_new_tokens):  
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond) 
            logits = logits[:,-1,:] # becomes (B,C)
            probs = F.softmax(logits,dim=-1)
            next_index = torch.multinomial(probs,num_samples=1) # (B,1)
            index = torch.cat((index,next_index),dim=1) # (B,T+1) 
        return index
    
model = GPTLanguageModel(vocab_size)
print("Loading Model...")
with open('../models/model_01.pkl','rb') as f:
    model = pickle.load(f)
print("Model Loaded Successfully!")
m = model.to(device)


Loading Model...
Model Loaded Successfully


In [13]:
@torch.no_grad() 
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [14]:
optimizer = torch.optim.AdamW(model.parameters(),learning_rate)

for iters in range(max_iters):
    if iters % eval_iters == 0:
        losses = estimate_loss()
        print(f"Iteration : {iters}  Train Loss : {losses['train']:.4f}  Val Loss : {losses['val']:.4f}")

    xb,yb = get_batch('train')

    # evaluate loss
    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

with open('../models/model_01.pkl','wb') as f:
    pickle.dump(model,f)
print("model saved")

Iteration : 0  Train Loss : 4.4836  Val Loss : 4.4870
Iteration : 200  Train Loss : 2.4094  Val Loss : 2.4727
Iteration : 400  Train Loss : 2.2033  Val Loss : 2.2789
Iteration : 600  Train Loss : 2.0808  Val Loss : 2.1613
Iteration : 800  Train Loss : 2.0200  Val Loss : 2.1024
Iteration : 1000  Train Loss : 1.9591  Val Loss : 2.0460
Iteration : 1200  Train Loss : 1.9150  Val Loss : 2.0069
Iteration : 1400  Train Loss : 1.8953  Val Loss : 1.9884
Iteration : 1600  Train Loss : 1.8344  Val Loss : 1.9314
Iteration : 1800  Train Loss : 1.8259  Val Loss : 1.9087
Iteration : 2000  Train Loss : 1.8211  Val Loss : 1.9228
Iteration : 2200  Train Loss : 1.7869  Val Loss : 1.8769
Iteration : 2400  Train Loss : 1.7532  Val Loss : 1.8414
Iteration : 2600  Train Loss : 1.7399  Val Loss : 1.8431
Iteration : 2800  Train Loss : 1.7256  Val Loss : 1.8329
1.7796268463134766
model saved


In [ ]:
# As you can see the gpt model can predict characters with more accuracy and the text makes sense to some extent. As we train it with larger parameters it will work with higher accuracy, the model will predict almost everything correctly.
context = torch.zeros(size=(1,1),dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,500)[0].tolist())
print(generated_chars)


boo; "but wurt troad.

Doothy, ans wern of nind to seawi n tely eata tear abain, andly etreple--onath meaned,e, so in thly,eee nive goable st dame uc. ut lorvetty. Surry sort can eome; 'ne aress fisteds 'no any ofan wolle gera city
an  is
decaus itter.

"NiAPLER  Wave nist,me toum, abe again inves  said con't  their,'s sain the Dothy, withled, in.
"That  the sisife to bya so
lot,"a we a d. Her scible doon to staile ent's we
notening heward,. ert, and tookink. it at pickly instedents."

"Anic.

"


#### CHATBOT

In [ ]:
prompt = input("Prompt : ")
context = torch.tensor([encode(prompt)],dtype=torch.long,device=device)
generated_chars = decode(m.generate(context,200)[0].tolist())
print(generated_chars)

Hicabouted Jim hed told a ath leash siry unny walkina'p thath, eyed and eable the cittenued me excurceaun e couce bove to cences  wered raugina like anios, any easil roite wro---was  eand a
fotent alv
s
